In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# These imports assume your local modules are set up as below.
# Adjust them to match your actual file structure if needed.
from multi_llm_debate.analysis.correct_rate_by_round import (
    calculate_correct_rate_by_round,
    calculate_majority_vote_correct_rate_for_round_n
)
from multi_llm_debate.analysis.calculate_task_accuracy import analyze_task_accuracy


# ==========================
# 1. CONFIGURATION
# ==========================
DATA_PATH = Path("../output/bool_q/processed_data.csv")
MODEL_DIR_PATH = Path("../data/bool_q")

# Accuracy cutoffs for easy/medium/hard
EASY_THRESHOLD = 0.7
MEDIUM_THRESHOLD = 0.1

# We only want directories whose parentheses-based digit sum == 6
TARGET_MODEL_COUNT = 7

# Maximum round number for your correct_rate_by_round function
MAX_ROUND_NUMBER = 10


# ==========================
# 2. HELPER FUNCTIONS
# ==========================
def get_total_model_count(dir_name: str) -> int:
    """
    Parses folder name (e.g., 'llama3(3)+mistral(3)') 
    to sum up the numeric values found in parentheses.
    """
    # Find all digits inside parentheses: e.g. "(3)", "(6)"
    matches = re.findall(r"\((\d+)\)", dir_name)
    # Convert each match to an integer and sum
    return sum(int(m) for m in matches)


def create_plot(
    easy_rates, medium_rates, hard_rates, overall_rates,
    easy_mv_rate, medium_mv_rate, hard_mv_rate, overall_mv_rate,
    model_name
):
    """
    Plots round-by-round correct rates for easy, medium, hard, and overall tasks,
    along with corresponding majority-vote horizontal lines.
    """
    # Define the colors for the plot
    colors = {
        "easy": 'g',     # Green for Easy
        "medium": 'y',   # Yellow for Medium
        "hard": 'r',     # Red for Hard
        "overall": 'b',  # Blue for Overall
    }
    
    # Determine how many rounds we have data for
    max_rounds = max(
        len(easy_rates),
        len(medium_rates),
        len(hard_rates),
        len(overall_rates)
    )
    rounds = range(max_rounds)

    plt.figure(figsize=(10, 6))

    # Plot lines only if there's data
    if len(easy_rates) > 0:
        plt.plot(rounds, easy_rates, f'{colors["easy"]}-o', label='Easy Tasks', linewidth=2)
    if len(medium_rates) > 0:
        plt.plot(rounds, medium_rates, f'{colors["medium"]}-s', label='Medium Tasks', linewidth=2)
    if len(hard_rates) > 0:
        plt.plot(rounds, hard_rates, f'{colors["hard"]}-d', label='Hard Tasks', linewidth=2)
    if len(overall_rates) > 0:
        plt.plot(rounds, overall_rates, f'{colors["overall"]}-*', label='Overall', linewidth=2)

    # Horizontal lines for majority vote rates
    plt.axhline(y=easy_mv_rate, color=colors["easy"], linestyle='--', label='Easy Majority Vote')
    plt.axhline(y=medium_mv_rate, color=colors["medium"], linestyle='--', label='Medium Majority Vote')
    plt.axhline(y=hard_mv_rate, color=colors["hard"], linestyle='--', label='Hard Majority Vote')
    plt.axhline(y=overall_mv_rate, color=colors["overall"], linestyle='--', label='Overall Majority Vote')

    # Construct the chart title
    plt.title(
        f'Correct Rate by Round: {model_name}\n'
        f'(Easy >= {EASY_THRESHOLD}, '
        f'{MEDIUM_THRESHOLD} <= Medium < {EASY_THRESHOLD}, '
        f'Hard < {MEDIUM_THRESHOLD})',
        pad=15
    )
    plt.xlabel('Round Number')
    plt.ylabel('Correct Rate')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()

    # Set y-axis limits and ticks
    plt.ylim(0, 1)
    plt.yticks(np.arange(0, 1.1, 0.1))
    plt.xticks(rounds)

    plt.tight_layout()
    plt.show()


def process_model(model_dir: Path):
    """
    Performs the entire pipeline for a given model directory:
    1) Analyze accuracy,
    2) Split tasks by easy/medium/hard using "accuracy" column,
    3) Compute correct rates by round,
    4) Compute majority-vote correctness,
    5) Generate the plot.
    """
    model_name = model_dir.name  # The directory name, e.g. "llama3(3)+mistral(3)"
    print(f"\nProcessing model: {model_name}")

    # -------------------------------------------------------------
    # 1. (Optional) Analyze and update the dataframe with accuracy 
    #    if your pipeline needs it. Otherwise, skip this step.
    # -------------------------------------------------------------
    result_df = analyze_task_accuracy(
        model_dir=model_dir,
        dataframe=df,  # Global DataFrame loaded below
    )

    # -------------------------------------------------------------
    # 2. Categorize tasks by "accuracy": easy / medium / hard
    # -------------------------------------------------------------
    easy_df = result_df[result_df["accuracy"] >= EASY_THRESHOLD]
    medium_df = result_df[
        (result_df["accuracy"] < EASY_THRESHOLD) & (result_df["accuracy"] >= MEDIUM_THRESHOLD)
    ]
    hard_df = result_df[result_df["accuracy"] < MEDIUM_THRESHOLD]

    # -------------------------------------------------------------
    # 3. Calculate correct rates by round
    # -------------------------------------------------------------
    easy_cr = calculate_correct_rate_by_round(easy_df, model_dir, max_round_number=MAX_ROUND_NUMBER)
    medium_cr = calculate_correct_rate_by_round(medium_df, model_dir, max_round_number=MAX_ROUND_NUMBER)
    hard_cr = calculate_correct_rate_by_round(hard_df, model_dir, max_round_number=MAX_ROUND_NUMBER)
    overall_cr = calculate_correct_rate_by_round(result_df, model_dir, max_round_number=MAX_ROUND_NUMBER)

    # Extract the row where 'metric' == 'absolute' and then pull columns from round_1 .. round_N
    def get_rates(df_cr: pd.DataFrame):
        if df_cr.empty:
            return []
        row_abs = df_cr[df_cr['metric'] == 'absolute']
        if row_abs.empty:
            return []
        return row_abs.iloc[0, 2:].values  # skipping columns "metric" and "model_dir"

    easy_rates = get_rates(easy_cr)
    medium_rates = get_rates(medium_cr)
    hard_rates = get_rates(hard_cr)
    overall_rates = get_rates(overall_cr)

    # -------------------------------------------------------------
    # 4. Calculate majority-vote correctness
    # -------------------------------------------------------------
    easy_mv_rate = ( 
        calculate_majority_vote_correct_rate_for_round_n(easy_df, model_dir) 
        if not easy_df.empty else 0.0
    )
    medium_mv_rate = (
        calculate_majority_vote_correct_rate_for_round_n(medium_df, model_dir)
        if not medium_df.empty else 0.0
    )
    hard_mv_rate = (
        calculate_majority_vote_correct_rate_for_round_n(hard_df, model_dir)
        if not hard_df.empty else 0.0
    )
    overall_mv_rate = calculate_majority_vote_correct_rate_for_round_n(result_df, model_dir)

    # -------------------------------------------------------------
    # 5. Create the plot
    # -------------------------------------------------------------
    create_plot(
        easy_rates, medium_rates, hard_rates, overall_rates,
        easy_mv_rate, medium_mv_rate, hard_mv_rate, overall_mv_rate,
        model_name
    )


# ==========================
# 3. MAIN SCRIPT
# ==========================
if __name__ == "__main__":
    # Load your main data
    df = pd.read_csv(DATA_PATH)

    # Get all model directories
    all_model_dirs = list(MODEL_DIR_PATH.glob('*'))

    # Filter directories by total model count == TARGET_MODEL_COUNT
    filtered_model_dirs = [
        d for d in all_model_dirs
        if get_total_model_count(d.name) == TARGET_MODEL_COUNT
    ]

    print("Filtered directories (sum of parentheses == 6):")
    for d in filtered_model_dirs:
        print("  -", d.name)

    # Now, process each filtered directory
    for model_dir in filtered_model_dirs:
        process_model(model_dir)


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# These imports assume your local modules are set up as below.
# Adjust them to match your actual file structure if needed.
from multi_llm_debate.analysis.correct_rate_by_round import (
    calculate_correct_rate_by_round,
)
from multi_llm_debate.analysis.calculate_task_accuracy import analyze_task_accuracy


# ==========================
# 1. CONFIGURATION
# ==========================
DATA_PATH = Path("../output/bool_q/processed_data.csv")
MODEL_DIR_PATH = Path("../data/bool_q")

# We only want directories whose parentheses-based digit sum == 6
TARGET_MODEL_COUNT = 7

# Maximum round number for your correct_rate_by_round function
MAX_ROUND_NUMBER = 10


# ==========================
# 2. HELPER FUNCTIONS
# ==========================
def get_total_model_count(dir_name: str) -> int:
    """
    Parses folder name (e.g., 'llama3(3)+mistral(3)') 
    to sum up the numeric values found in parentheses.
    """
    matches = re.findall(r"\((\d+)\)", dir_name)
    return sum(int(m) for m in matches)


def create_plot(accuracies_by_round, model_name):
    """
    Plots a line for each unique accuracy value from the correct_rate_by_round results,
    with the legend sorted by ascending accuracy values.
    """
    # Sort the dictionary items by the accuracy value (the dictionary key)
    sorted_items = sorted(accuracies_by_round.items(), key=lambda x: x[0])
    
    # Use a color map, sized to the number of unique accuracy lines
    colors = plt.cm.get_cmap('tab20', len(sorted_items))

    plt.figure(figsize=(10, 6))

    # Plot a line for each unique accuracy value, in descending order
    for idx, (accuracy, accuracy_values) in enumerate(sorted_items[::-1]):
        rounds = range(len(accuracy_values))
        plt.plot(
            rounds,
            accuracy_values,
            label=f'Accuracy = {accuracy:.2f}',
            color=colors(idx),
            linewidth=2,
        )

    # Title and labels
    plt.title(f'Accuracy by Round: {model_name}', pad=15)
    plt.xlabel('Round Number')
    plt.ylabel('Correct Rate')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()

    # Set y-axis limits and ticks
    plt.ylim(0, 1)
    plt.yticks(np.arange(0, 1.1, 0.1))
    plt.xticks(range(MAX_ROUND_NUMBER))

    plt.tight_layout()
    plt.show()


def process_model(model_dir: Path):
    """
    1) Analyze accuracy,
    2) Calculate correct rates for each unique accuracy value,
    3) Print the percentage of tasks for each accuracy value,
    4) Plot the results.
    """
    model_name = model_dir.name
    print(f"\nProcessing model: {model_name}")

    # 1) Analyze accuracy
    result_df = analyze_task_accuracy(
        model_dir=model_dir,
        dataframe=df  # Global DataFrame loaded from main
    )

    # 2) Get all unique accuracy values from the result dataframe
    unique_accuracies = result_df['accuracy'].unique()

    # 3) Create a dictionary to store the accuracy values by round
    accuracies_by_round = {accuracy: [] for accuracy in unique_accuracies}

    # 4) For each unique accuracy value, calculate the fraction of tasks per round
    total_tasks = len(result_df)  # Total number of tasks in result_df
    for accuracy in unique_accuracies:
        if accuracy < 0:
            continue
        # Filter tasks by accuracy
        filtered_df = result_df[result_df['accuracy'] == accuracy]
        
        # Calculate and print the percentage of tasks with this accuracy
        accuracy_percentage = (len(filtered_df) / 2000) * 100
        print(f"Accuracy = {accuracy:.2f}: {accuracy_percentage:.2f}% of total tasks")

        # Calculate correct rates for this accuracy using calculate_correct_rate_by_round
        cr_filtered_df = calculate_correct_rate_by_round(filtered_df, model_dir, max_round_number=MAX_ROUND_NUMBER)
        
        # Extract the correct rates for this accuracy value
        accuracies_by_round[accuracy] = cr_filtered_df[
            cr_filtered_df['metric'] == 'absolute'
        ].iloc[0, 2:].values

    # 5) Create the plot
    create_plot(accuracies_by_round, model_name)



# ==========================
# 3. MAIN SCRIPT
# ==========================
if __name__ == "__main__":
    # Load the main data
    df = pd.read_csv(DATA_PATH)

    # Get all model directories
    all_model_dirs = list(MODEL_DIR_PATH.glob('*'))

    # Filter directories by total model count == TARGET_MODEL_COUNT
    filtered_model_dirs = [
        d for d in all_model_dirs
        if get_total_model_count(d.name) == TARGET_MODEL_COUNT
    ]

    print("Filtered directories (sum of parentheses == 6):")
    for d in filtered_model_dirs:
        print("  -", d.name)

    # Process each filtered directory
    for model_dir in filtered_model_dirs:
        process_model(model_dir)


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from multi_llm_debate.analysis.calculate_correct_rate_distribution import (
    calculate_correct_rate_distribution_for_round_n,
)
import pandas as pd
import seaborn as sns

from typing import Dict, List, Tuple

def process_distribution_data(
    result_df: pd.DataFrame,
    round_number: int,
) -> Dict[str, float]:
    """Process distribution data to get percentages for each bin.
    
    Args:
        result_df: DataFrame with distribution data.
        round_number: The round number being processed.
        
    Returns:
        Dictionary mapping bin labels to percentages.
    """
    # Get bin columns (those that are digits)
    bin_columns = [col for col in result_df.columns if col.isdigit()]
    bin_columns.sort(key=int)  # Sort numerically
    
    if not bin_columns or result_df.empty:
        print(f"No bins found for round {round_number}")
        return {}
    
    # Count tasks and calculate percentages
    task_count = len(result_df)
    bin_sums = result_df[bin_columns].sum()
    bin_percentages = (bin_sums / task_count * 100).to_dict()
    
    return bin_percentages

def plot_round_distribution(
    bin_percentages: Dict[str, float],
    round_number: int,
    output_dir: Path,
    show_plot: bool = False,
) -> None:
    """Create and save a plot of the correct rate distribution for a round.
    
    Args:
        bin_percentages: Dictionary mapping bin labels to percentage values.
        round_number: The round number being visualized.
        output_dir: Directory where the plot should be saved.
        show_plot: Whether to display the plot interactively.
    """
    if not bin_percentages:
        print(f"No data to plot for round {round_number}")
        return
    
    plt.figure(figsize=(10, 6))
    
    # Sort bins numerically
    bins = [int(b) for b in sorted(bin_percentages.keys(), key=int)]
    values = [bin_percentages[str(b)] for b in bins]
    
    # Create bar chart
    bars = plt.bar(bins, values)
    
    # Add value labels on top of bars
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2.,
            height + 1,
            f'{height:.1f}%',
            ha='center',
            fontsize=9,
        )
    
    # Set chart attributes
    plt.title(f'Round {round_number}: Distribution of Correct Agents', 
             fontsize=14)
    plt.xlabel('Number of Correct Agents', fontsize=12)
    plt.ylabel('Percentage of Tasks (%)', fontsize=12)
    plt.grid(axis='y', alpha=0.3)
    plt.ylim(0, max(values) * 1.2)  # Add some headroom for labels
    
    # Save the plot
    output_path = output_dir / f"round_{round_number}_distribution.png"
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    
    if show_plot:
        plt.show()
    plt.close()
    
    print(f"Saved plot for round {round_number} to {output_path}")

def create_heatmap(
    all_distributions: List[Tuple[int, Dict[str, float]]],
    output_dir: Path,
) -> None:
    """Create a heatmap showing the evolution of distributions across rounds.
    
    Args:
        all_distributions: List of (round_number, bin_percentages) tuples.
        output_dir: Directory where the plot should be saved.
    """
    if not all_distributions:
        print("No data to create heatmap")
        return
    
    # Create a DataFrame from the collected data
    data = []
    for round_num, bin_percentages in all_distributions:
        for bin_label, percentage in bin_percentages.items():
            data.append({
                'Round': round_num,
                'Correct Agents': int(bin_label),
                'Percentage': percentage
            })
    
    df = pd.DataFrame(data)
    
    # Create pivot table for heatmap
    pivot_df = df.pivot(
        index='Round', 
        columns='Correct Agents', 
        values='Percentage'
    ).fillna(0)
    
    # Create heatmap plot
    plt.figure(figsize=(12, 8))
    ax = sns.heatmap(
        pivot_df,
        annot=True,
        fmt=".1f",
        cmap="YlGnBu",
        linewidths=0.5,
        cbar_kws={'label': 'Percentage of Tasks (%)'}
    )
    
    plt.title('Evolution of Correct Agent Distribution Across Rounds', 
             fontsize=16)
    plt.tight_layout()
    
    # Save the heatmap
    # output_path = output_dir / "correct_agent_distribution_heatmap.png"
    # plt.savefig(output_path, dpi=300)
    plt.show()  # Show the heatmap interactively
    plt.close()
    
    # print(f"Saved heatmap to {output_path}")

def main(
    data_path: Path,
    model_dir: Path,
    output_dir: Path,
    max_rounds: int = 6,
    show_plots: bool = False,
) -> None:
    """Run the visualization process for correct rate distributions.
    
    Args:
        data_path: Path to the CSV file with task data.
        model_dir: Directory containing model output data.
        output_dir: Directory where output plots should be saved.
        max_rounds: Maximum number of rounds to process.
        show_plots: Whether to display plots interactively.
    """
    # Create output directory if it doesn't exist
    output_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        # Load data
        dataframe = pd.read_csv(data_path)
        print(f"Loaded data from {data_path}")
    except Exception as e:
        print(f"Error loading data: {e}")
        return
    
    all_distributions = []
    
    # Process each round
    for round_number in range(max_rounds):
        print(f"Processing round {round_number}...")
        
        # Calculate distribution
        result_df = calculate_correct_rate_distribution_for_round_n(
            dataframe=dataframe,
            model_dir=model_dir,
            round_number=round_number,
        )
        
        # Process and store the distribution data
        bin_percentages = process_distribution_data(result_df, round_number)
        
        if bin_percentages:
            all_distributions.append((round_number, bin_percentages))
            
            # Create individual round plot
            plot_round_distribution(
                bin_percentages,
                round_number,
                output_dir,
                show_plot=show_plots,
            )
    
    # Create summary heatmap visualization
    create_heatmap(all_distributions, output_dir)
    
    print("Visualization complete!")
    
    
    
if __name__ == "__main__":
    # Define paths
    DATA_PATH = Path("../output/bool_q/processed_data.csv")
    MODEL_DIR_PATH = Path("../data/bool_q/llama3(11)")
    OUTPUT_DIR = Path("../output")

    # Run the main function
    main(
        data_path=DATA_PATH,
        model_dir=MODEL_DIR_PATH,
        output_dir=OUTPUT_DIR,
        max_rounds=6,  # Adjust as needed
        show_plots=True,  # Set to True to display plots interactively
    )